# Swin Transformer 

## Patch Embedding

* Patch Embedding é a primeira etapa da rede, responsável por transformar a imagem de entrada (pixels brutos) em uma sequência de "tokens" que o Transformer consegue processar.

In [ ]:
import torch
import torch.nn as nn


class PatchEmbedding(nn.Module):

    def __init__(
        self,
        in_channels=1,
        embed_dim=96,  # cada patch será transformado em um vetor de 96 dimensões
        patch_size=4
    ):
        super().__init__()

        self.proj = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embed_dim,   
            # filtro convolucional "olha" para blocos de 4×4 pixels
            kernel_size=patch_size,
            # pula 4 pixels a cada passo, evitando sobreposição de patches
            stride=patch_size 
        )

    def forward(self, x):

        # x = (B, C_in, H, W) = (2, 1, 256, 256)

        x = self.proj(x)           # (B, embed_dim, H/4, W/4)
        x = x.permute(0, 2, 3, 1)  # permute reorganiza a dimensão para: (B, H/4, W/4, embed_dim)

        return x

In [3]:
# EXEMPLO

x = torch.randn(2, 1, 256, 256)

patch_embed = PatchEmbedding(
    in_channels=1,
    embed_dim=96,
    patch_size=4
)

y = patch_embed(x)

print(x.shape)
# cada patch agora é representado por um vetor de 96 números
print(y.shape)

torch.Size([2, 1, 256, 256])
torch.Size([2, 64, 64, 96])


## Window Partition

* Depois do Patch Embedding, temos um mapa de features tipo (B, H, W, C);
* O Window Partition simplesmente corta o mapa de features em blocos (janelas) não sobrepostos de tamanho fixo, assim a atenção pode ser calculada independentemente dentro de cada janela..

In [4]:
def window_partition(x, window_size):

    B, H, W, C = x.shape

    x = x.view(
        B,
        H // window_size,
        window_size,
        W // window_size,
        window_size,
        C
    )

    windows = x.permute(
        0, 1, 3, 2, 4, 5
    ).contiguous()

    windows = windows.view(
        -1,
        window_size,
        window_size,
        C
    )

    return windows

In [ ]:
# EXEMPLO - de [2, 64, 64, 96] para [128, 8, 8, 96]

    # B = 2

    # 64 × 64
    #    ↓
    # 8 × 8 windows
    #    ↓
    # 64 windows/imagem

    # 2 × 64 = 128 windows

    # cada window:
    # 8 × 8 × 96

x = torch.randn(2, 64, 64, 96)

windows = window_partition(
    x,
    window_size=8
)

print(windows.shape)

torch.Size([128, 8, 8, 96])


## Window Reverse

* O window_reverse é a operação inversa do window_partition.

In [6]:
def window_reverse(
    windows,
    window_size,
    H,
    W
):

    B = int(
        windows.shape[0]
        / (H * W / window_size / window_size)
    )

    x = windows.view(
        B,
        H // window_size,
        W // window_size,
        window_size,
        window_size,
        -1
    )

    x = x.permute(
        0, 1, 3, 2, 4, 5
    ).contiguous()

    x = x.view(
        B,
        H,
        W,
        -1
    )

    return x

In [7]:
# EXEMPLO

x = torch.randn(2, 64, 64, 96)

windows = window_partition(x, 8)

x_reconstructed = window_reverse(
    windows,
    8,
    64,
    64
)

print(x.shape)
print(x_reconstructed.shape)

torch.Size([2, 64, 64, 96])
torch.Size([2, 64, 64, 96])


## Window Attention (W-MSA)

* A Window Attention é o módulo que aplica self-attention (multi-head) dentro de cada janela, tratando os patches daquela janela como uma sequência de tokens.

*Sendo que:*

* Cada janela de window_size × window_size patches vira uma sequência de window_size²  tokens.
* Calcula-se self-attention normal (Q, K, V) dentro dessa sequência.
* Adiciona-se um relative position bias (viés de posição relativa), como uma particularidade importante do Swin.

In [12]:
# [B_ × N, C]

# B_ = número de janelas
# N  = 8 × 8 = 64 tokens por janela
# C  = 96 features

# [128, 64, 96]

class WindowAttention(nn.Module):

    def __init__(
        self,
        dim,
        window_size,
        num_heads
    ):
        super().__init__()

        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads

        head_dim = dim // num_heads

        self.scale = head_dim ** -0.5

        # Q, K e V
        self.qkv = nn.Linear(
            dim,
            dim * 3,
            bias=True
        )

        # Projeção depois da atenção
        self.proj = nn.Linear(
            dim,
            dim
        )

    def forward(self, x):

        # x:
        # [num_windows * B, N, C]

        B_, N, C = x.shape

        qkv = self.qkv(x)

        qkv = qkv.reshape(
            B_,
            N,
            3,
            self.num_heads,
            C // self.num_heads
        )

        qkv = qkv.permute(
            2, 0, 3, 1, 4
        )

        q, k, v = qkv.unbind(0)

        # Escalação
        q = q * self.scale

        # Attention Score
        attn = q @ k.transpose(-2, -1)

        # Softmax
        attn = attn.softmax(dim=-1)

        # Aplicação sobre V - multiplicação matricial
        x = attn @ v

        # Voltar para [B_, N, C]
        x = x.transpose(1, 2)

        x = x.reshape(
            B_,
            N,
            C
        )

        # Projeção final
        x = self.proj(x)

        return x

In [13]:
# EXEMPLO

x = torch.randn(
    128,
    64,
    96
)

attention = WindowAttention(
    dim=96,
    window_size=8,
    num_heads=4
)

y = attention(x)

print(y.shape)

torch.Size([128, 64, 96])


*OBS.: A atenção não altera a quantidade de tokens nem a dimensão dos embeddings.*

## Relative Position Bias

O Swin em vez de dizer a posição exata de cada pedaço da imagem, ele avisa o quão longe um pedaço está do outro.

A lógica é, quanto mais perto ou mais longe o pedaço B estiver do pedaço A, mais (ou menos) atenção o modelo vai dar para essa relação.

### Código com o BIAS mapeado:

In [14]:
class WindowAttention(nn.Module):

    def __init__(
        self,
        dim,
        window_size,
        num_heads
    ):
        super().__init__()

        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads

        head_dim = dim // num_heads

        self.scale = head_dim ** -0.5

        # ------------------------------------------------
        # Relative Position Bias Table
        # ------------------------------------------------

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros(
                (2 * window_size - 1) *
                (2 * window_size - 1),
                num_heads
            )
        )

        # ------------------------------------------------
        # Relative Position Index
        # ------------------------------------------------

        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)

        coords = torch.stack(
            torch.meshgrid(
                coords_h,
                coords_w,
                indexing="ij"
            )
        )

        # [2, Wh, Ww]
        coords_flatten = torch.flatten(
            coords,
            1
        )

        # [2, Wh*Ww]
        relative_coords = (
            coords_flatten[:, :, None]
            -
            coords_flatten[:, None, :]
        )

        # [2, Wh*Ww, Wh*Ww]
        relative_coords = relative_coords.permute(
            1,
            2,
            0
        ).contiguous()

        # deslocamento para valores positivos
        relative_coords[:, :, 0] += (
            window_size - 1
        )

        relative_coords[:, :, 1] += (
            window_size - 1
        )

        relative_coords[:, :, 0] *= (
            2 * window_size - 1
        )

        relative_position_index = (
            relative_coords.sum(-1)
        )

        self.register_buffer(
            "relative_position_index",
            relative_position_index
        )

        # ------------------------------------------------
        # QKV
        # ------------------------------------------------

        self.qkv = nn.Linear(
            dim,
            dim * 3
        )

        self.proj = nn.Linear(
            dim,
            dim
        )

        nn.init.trunc_normal_(
            self.relative_position_bias_table,
            std=0.02
        )

    def forward(self, x):

        B_, N, C = x.shape

        qkv = self.qkv(x)

        qkv = qkv.reshape(
            B_,
            N,
            3,
            self.num_heads,
            C // self.num_heads
        )

        qkv = qkv.permute(
            2,
            0,
            3,
            1,
            4
        )

        q, k, v = qkv.unbind(0)

        q = q * self.scale

        # ------------------------------------------------
        # Attention
        # ------------------------------------------------

        attn = q @ k.transpose(-2, -1)

        # ------------------------------------------------
        # Relative Position Bias
        # ------------------------------------------------

        relative_position_bias = (
            self.relative_position_bias_table[
                self.relative_position_index.view(-1)
            ]
        )

        relative_position_bias = (
            relative_position_bias.view(
                self.window_size *
                self.window_size,

                self.window_size *
                self.window_size,

                -1
            )
        )

        relative_position_bias = (
            relative_position_bias.permute(
                2,
                0,
                1
            ).contiguous()
        )

        attn = attn + relative_position_bias.unsqueeze(0)

        # ------------------------------------------------
        # Softmax
        # ------------------------------------------------

        attn = attn.softmax(dim=-1)

        # ------------------------------------------------
        # Attention × V
        # ------------------------------------------------

        x = attn @ v

        x = x.transpose(
            1,
            2
        ).reshape(
            B_,
            N,
            C
        )

        x = self.proj(x)

        return x

## Multi-Layer Perceptron (MPL)

O MLP é o "cérebro individual" de cada pedaço da imagem.

Enquanto a atenção faz os pedaços da imagem conversarem entre si, o MLP processa as informações de cada pedaço isoladamente para extrair características mais complexas.

O caminho que o dado faz é:

1. LayerNorm: Ajusta os valores para ficarem na mesma escala.
2. Linear (4×): Aumenta o tamanho da informação em 4 vezes para ter mais espaço de processamento.
3. GELU: Aplica a ativação (a matemática que dá o "poder de aprendizado" não-linear).
4. Linear (contrai): Encolhe a informação de volta ao tamanho original.
5. +Residual: Soma o resultado com o dado que entrou no início (para facilitar o aprendizado).

In [15]:
class MLP(nn.Module):

    def __init__(
        self,
        dim,
        hidden_dim,
        dropout=0.0
    ):
        super().__init__()

        self.fc1 = nn.Linear(
            dim,
            hidden_dim
        )

        self.act = nn.GELU()

        self.fc2 = nn.Linear(
            hidden_dim,
            dim
        )

        self.drop = nn.Dropout(
            dropout
        )

    def forward(self, x):

        x = self.fc1(x)

        x = self.act(x)

        x = self.drop(x)

        x = self.fc2(x)

        x = self.drop(x)

        return x

## Swin Transformer Block

O Swin Transformer Block é a estrutura principal que junta todas essas peças em uma etapa completa. Ele funciona como um bloco Transformer tradicional, com conexões residuais, normalização de dados e uma camada MLP mas adaptado para a imagem:

* Em vez de olhar a imagem toda de uma vez: ele processa apenas pequenas janelas.
* Como ele junta tudo: a cada bloco, ele alterna entre usar janelas normais (Regular Window) e janelas levemente movidas (Shifted Window).

Essa troca garante que o modelo não analise só os pedaços isolados, mas também aprenda como as bordas dessas janelas se conectam entre si.

In [16]:
class SwinTransformerBlock(nn.Module):

    def __init__(
        self,
        dim,
        num_heads,
        window_size=8,
        mlp_ratio=4.0
    ):
        super().__init__()

        self.dim = dim
        self.window_size = window_size

        # ---------------------------------------
        # Normalização
        # ---------------------------------------

        self.norm1 = nn.LayerNorm(dim)

        # ---------------------------------------
        # Window Attention
        # ---------------------------------------

        self.attn = WindowAttention(
            dim=dim,
            window_size=window_size,
            num_heads=num_heads
        )

        # ---------------------------------------
        # MLP
        # ---------------------------------------

        hidden_dim = int(
            dim * mlp_ratio
        )

        self.norm2 = nn.LayerNorm(dim)

        self.mlp = MLP(
            dim=dim,
            hidden_dim=hidden_dim
        )

    def forward(self, x):

        # x:
        # [B, H, W, C]

        B, H, W, C = x.shape

        shortcut = x

        # ---------------------------------------
        # LayerNorm
        # ---------------------------------------

        x = self.norm1(x)

        # ---------------------------------------
        # Window Partition
        # ---------------------------------------

        windows = window_partition(
            x,
            self.window_size
        )

        # [B*nW, Wh, Ww, C]
        # ↓
        # [B*nW, Wh*Ww, C]

        windows = windows.view(
            -1,
            self.window_size *
            self.window_size,
            C
        )

        # ---------------------------------------
        # Attention
        # ---------------------------------------

        attn_windows = self.attn(
            windows
        )

        # ---------------------------------------
        # Voltar para janelas 2D
        # ---------------------------------------

        attn_windows = attn_windows.view(
            -1,
            self.window_size,
            self.window_size,
            C
        )

        # ---------------------------------------
        # Window Reverse
        # ---------------------------------------

        x = window_reverse(
            attn_windows,
            self.window_size,
            H,
            W
        )

        # ---------------------------------------
        # Residual
        # ---------------------------------------

        x = shortcut + x

        # ---------------------------------------
        # MLP
        # ---------------------------------------

        x = x + self.mlp(
            self.norm2(x)
        )

        return x

In [17]:
# EXEMPLO - TESTANDO O SWIN TRANSFORMER

x = torch.randn(
    2,
    64,
    64,
    96
)

block = SwinTransformerBlock(
    dim=96,
    num_heads=4,
    window_size=8
)

y = block(x)

print("Entrada :", x.shape)
print("Saída   :", y.shape)

Entrada : torch.Size([2, 64, 64, 96])
Saída   : torch.Size([2, 64, 64, 96])


*OBS.: O Swin Block não reduz a resolução.*